# Evaluation

Den här notebooken utvärderar projektets metoder för kompetensidentifiering. Baseline utvärderas dels mot endast taxonomikompetenser, dels mot samtliga manuellt annoterade kompetenser. Den förbättrade NLP-metoden utvärderas mot samtliga annoterade kompetenser. Metoderna jämförs med precision, recall och F1-score samt genom felanalys av false positives och false negatives. Test-setet har inte använts vid utvecklingen av regler, synonymer eller modeller.

In [ ]:
import json
import re
import pandas as pd

In [ ]:
# läser in orginalets test-set
with open("../data/test_set.json", "r", encoding="utf-8") as file:
    test_ads = json.load(file)

test_df = pd.DataFrame(test_ads)

test_df.head()

In [ ]:
print("Antal annonser i test-set:", len(test_df))
print("Kolumner:", test_df.columns.tolist())

In [ ]:
# läser in det annoterade labelstudios test-set
with open(
    "../data/annoterad_test_data.json",
    "r",
    encoding="utf-8"
) as file:
    annotated_data = json.load(file)

In [ ]:
# undersöker strukturen på labelstudios data
print(type(annotated_data))
print("Antal poster:", len(annotated_data))

In [ ]:
annotated_data[0]

In [ ]:
annotated_data[0].keys()

In [ ]:
for key, value in annotated_data[0].items():
    print(key, ":", type(value))

In [27]:
# bygger två gold-standard:
# 1. alla annoterade skills
# 2. endast skills märkta som Taxonomi

def extract_gold_annotations(annotated_data):
    gold_all = {}
    gold_taxonomy = {}

    for task in annotated_data:

        ad_id = str(task["data"]["id"])

        all_skills = set()
        taxonomy_skills = set()

        for annotation in task.get("annotations", []):

            for result in annotation.get("result", []):

                value = result.get("value", {})

                text = value.get("text")
                labels = value.get("labels", [])

                if not text:
                    continue

                # Alla annoterade skills
                if (
                    "Taxonomi" in labels
                    or "New skill" in labels
                ):
                    all_skills.add(text)

                # Endast skills från taxonomin
                if "Taxonomi" in labels:
                    taxonomy_skills.add(text)

        gold_all[ad_id] = all_skills
        gold_taxonomy[ad_id] = taxonomy_skills

    return gold_all, gold_taxonomy

In [28]:
gold_all_raw, gold_taxonomy_raw = extract_gold_annotations(
    annotated_data
)

In [29]:
print("Exempel - alla skills:")
print(list(gold_all_raw.items())[:3])

print("\nExempel - endast Taxonomi:")
print(list(gold_taxonomy_raw.items())[:3])

Exempel - alla skills:
[('29389255', {'Databricks', 'Azure', 'Dev-Ops-området', ' Azure'}), ('29370869', {'svenska', 'kravarbete inom IT-lösningar', 'engelska'}), ('29392086', {'analysverktyg', 'datamodellering', 'Pega Business Architect (BA)', 'svenska', 'ERP', 'CRM', 'Pega Business Architect (PCBA) ', 'kravställa Pega Constellation', 'Oracle databaser', 'kravanalytiker', 'datamodeller', 'Pega', 'SQL', 'BIX-teknologi i Pega'})]

Exempel - endast Taxonomi:
[('29389255', {' Azure', 'Azure'}), ('29370869', {'kravarbete inom IT-lösningar'}), ('29392086', {'analysverktyg', 'datamodellering', 'ERP', 'CRM', 'Oracle databaser', 'kravanalytiker', 'datamodeller'})]


In [30]:
# kollar upp att ID:n matchar mellan test-set och gold standard

test_ids = set(
    test_df["id"].astype(str)
)

gold_ids = set(
    gold_all_raw.keys()
)

print("Test-set IDs:", len(test_ids))
print("Gold IDs:", len(gold_ids))

print(
    "Saknas i gold:",
    len(test_ids - gold_ids)
)

print(
    "Saknas i test_set:",
    len(gold_ids - test_ids)
)

Test-set IDs: 192
Gold IDs: 192
Saknas i gold: 0
Saknas i test_set: 0


In [31]:
# läser in skills som används av baseline

with open(
    "../data/skills_2511.json",
    "r",
    encoding="utf-8"
) as file:
    skills_data = json.load(file)

skill_names = [
    item["skill"]
    for item in skills_data
]

print("Antal skills i baseline:", len(skill_names))

Antal skills i baseline: 204


In [32]:
# den slutliga baseline-funktionen från notebook 04
def baseline_extract_skills(text, skills):
    found_skills = []

    text = text.lower()

    for skill in skills:

        pattern = (
            r"(?<!\w)"
            + re.escape(skill.lower())
            + r"(?!\w)"
        )

        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills

In [33]:
# kör på test-set
test_df["baseline_skills"] = test_df["description"].apply(
    lambda text: baseline_extract_skills(
        text,
        skill_names
    )
)

In [34]:
test_df[
    ["id", "baseline_skills"]
].head()

,id,baseline_skills
0,29389255,[]
1,29370869,[]
2,29392086,[Datamodellering]
3,29413273,[]
4,29377922,[]


In [35]:
# gör dictionary
baseline_predictions = {
    str(row["id"]): set(row["baseline_skills"])
    for _, row in test_df.iterrows()
}